# WTI/Brent — full research run

**One notebook. Run All. Everything else is in the library.**

Works from a clone, from the project zip, or on Colab: the bootstrap cell
below finds the library and installs the dependencies itself.

This is the only notebook you need to touch. It fetches the data, fits every
model, runs the rolling out-of-sample ablation, applies multiple-comparison
control, checks robustness, evaluates a late development/validation split, backtests the tails, and
finishes with an automated verdict against the pre-specified v1.0 decision rule.

The other notebooks are supporting material: `00`/`01` are provenance,
`02` is the validation walkthrough, `03` is the state-space deep dive.

### The question

Does a scenario generator built from empirical innovations, half-life flexible
probabilities, macro-state conditioning, cointegration-aware dynamics and a
filtered common-trend state space produce better calibrated multi-horizon
distributions for WTI, Brent and their spread than a driftless random walk — and
is any difference larger than the sampling noise implied by overlapping forecast
windows and by the multiplicity of the model grid?

### Sections

| § | What it answers |
|---|---|
| 0 | Bootstrap: find the library, install dependencies |
| 1 | Configuration |
| 2 | Data, and how badly front-month splicing contaminates it |
| 3 | What kind of system is this? Johansen, VECM, state space, variance decomposition |
| 3b | Is the cointegrating relation stable across the 2015 export-ban repeal? |
| 4 | The rolling ablation |
| 5 | Leaderboard and pairwise significance |
| 6 | Multiplicity: Model Confidence Set and Romano–Wolf |
| 7 | Robustness: does a handful of origins decide the ranking? |
| 8 | Late development/validation split |
| 8b | Does the instability matter for forecasting? |
| 9 | Calibration and tail backtests |
| 10 | Verdict against the pre-specified v1.0 rule |
| 11 | Does WTI curve / Cushing state change spread mean reversion? |
| 12 | How to run the genuine post-2025 forward block |

## 0. Bootstrap

Run this first. It locates the `oil_futures_regime` library and installs whatever
is missing, whether you are on Colab, on a plain Jupyter install, or inside the
cloned repository.

**On Colab**: run the cell and, when prompted, upload
`oil-futures-regime-scenario-forecasting-v10.zip`. Nothing else is required.
Fill in `REPO_URL` once the repository is public and even that step disappears.

In [ ]:
# ---------------------------------------------------------------------------
# Environment bootstrap: locate the library, install dependencies.
# ---------------------------------------------------------------------------
import glob, importlib, importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = ""          # e.g. "https://github.com/<you>/oil-futures-regime-scenario-forecasting.git"
PKG = "oil_futures_regime"
IN_COLAB = ("google.colab" in sys.modules) or Path("/content").exists()
SEARCH_DIRS = [".", "/content", "/content/drive/MyDrive"]


def _pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0 and "externally-managed-environment" in (r.stderr or ""):
        # Debian/Ubuntu PEP 668 environments need an explicit override.
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "--break-system-packages", *pkgs],
                           capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  warning: could not install {pkgs}; install them manually if a "
              f"later cell fails\n  {(r.stderr or '').strip().splitlines()[-1:]}")


def _find_root(*extra_dirs):
    """Return the repo root that contains src/oil_futures_regime, if any."""
    here = Path.cwd()
    for base in [here, *list(here.parents)[:3]]:
        if (base / "src" / PKG / "__init__.py").exists():
            return base.resolve()
    for d in [*SEARCH_DIRS, *extra_dirs]:
        d = Path(d)
        if not d.exists():
            continue
        hits = glob.glob(str(d / "**" / "src" / PKG / "__init__.py"), recursive=True)
        if hits:
            return Path(hits[0]).parents[2].resolve()
    return None


def _unzip_any():
    """Extract the first archive that looks like this repository."""
    for d in SEARCH_DIRS:
        d = Path(d)
        if not d.exists():
            continue
        for z in sorted(d.glob("*.zip")):
            try:
                with zipfile.ZipFile(z) as zf:
                    if any(f"src/{PKG}/__init__.py" in n for n in zf.namelist()):
                        target = Path("/content" if IN_COLAB else ".") / "_repo"
                        zf.extractall(target)
                        print(f"extracted {z.name} -> {target}")
                        return target
            except zipfile.BadZipFile:
                continue
    return None


ROOT = _find_root()

if ROOT is None and REPO_URL:
    dest = Path("/content" if IN_COLAB else ".") / "oil-futures-repo"
    if not dest.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(dest)], check=False)
    ROOT = _find_root(dest)

if ROOT is None:
    _unzip_any()
    ROOT = _find_root()

if ROOT is None and IN_COLAB:
    print("Upload the project zip (oil-futures-regime-scenario-forecasting-v10.zip):")
    from google.colab import files
    files.upload()
    _unzip_any()
    ROOT = _find_root()

if ROOT is None:
    raise RuntimeError(
        "Could not find the library. Either (a) set REPO_URL above, "
        "(b) upload/place the project zip next to this notebook and re-run this "
        "cell, or (c) run the notebook from inside the cloned repository."
    )

sys.path.insert(0, str(ROOT / "src"))
print(f"repo root: {ROOT}")

# yfinance is only needed for a real-data run; skip it in synthetic mode.
# --- dependencies ----------------------------------------------------------
REQUIRED = {
    "numpy": "numpy>=1.26", "pandas": "pandas>=2.2", "scipy": "scipy>=1.11",
    "matplotlib": "matplotlib>=3.8", "statsmodels": "statsmodels>=0.14",
    "arch": "arch>=6.3", "pyarrow": "pyarrow>=15.0", "yfinance": "yfinance>=0.2",
    "xlrd": "xlrd>=2.0", "tabulate": "tabulate>=0.9",
}
missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    _pip(*missing)
    importlib.invalidate_caches()

import oil_futures_regime as _ofr
print(f"oil_futures_regime {_ofr.__version__} ready")

## 1. Configuration

Everything you might want to change lives in this one cell.

In [ ]:
import json, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
# ROOT and sys.path were set by the bootstrap cell above.

# ----------------------------------------------------------------------------
# Run mode
# ----------------------------------------------------------------------------
USE_SYNTHETIC = False   # True = simulated data with a known DGP, no network needed
FAST          = False   # True = 6 models, 2 horizons; use it to check the wiring
SYNTHETIC_BETA_BREAK = False   # synthetic mode only: plant a repeal-shaped break

# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
START, END, ANCHOR = "2013-01-01", "2025-03-08", "W-FRI"
OIL_TICKERS   = {"WTI": "CL=F", "BRENT": "BZ=F"}
STATE_TICKERS = {"VIX": "^VIX", "DXY": "DX-Y.NYB", "TNX": "^TNX"}

# ----------------------------------------------------------------------------
# Validation design
# ----------------------------------------------------------------------------
HORIZONS      = [1, 4, 12, 20] if not FAST else [1, 4]
OOS_START     = "2018-01-01"
STRIDE        = 4            # weeks between forecast origins
N_SIM         = 2_000 if not FAST else 500
HALF_LIFE     = 52           # weeks
MIN_MACRO_ESS_FRACTION = 0.35
BASELINE      = "RW_equal"
SEED          = 2026

# ----------------------------------------------------------------------------
# Inference
# ----------------------------------------------------------------------------
N_BOOT        = 2_000 if not FAST else 400
MCS_ALPHA     = 0.10         # 90% Model Confidence Set
VALIDATION_START = "2023-01-01" # late ranking-stability split; NOT an untouched holdout
CRISIS_WINDOW = ("2020-02-01", "2020-06-30")

# ----------------------------------------------------------------------------
# Stability of the cointegrating relation
# ----------------------------------------------------------------------------
BREAK_DATE       = "2015-12-18"   # US crude-oil export ban repealed
STABILITY_START  = "2008-01-01"   # dedicated longer sample; core forecast sample stays 2013+
ROLLING_WINDOW   = 156            # weeks, for the rolling-window challenger
N_BOOT_STABILITY = 499 if not FAST else 99
RUN_COMMODITY_STATE = False        # experimental EIA WTI C1-C4 + lagged Cushing branch

OUT = ROOT / "reports" / ("oos_synthetic" if USE_SYNTHETIC else "oos")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
print(f"mode: {'SYNTHETIC' if USE_SYNTHETIC else 'REAL DATA'} | fast: {FAST} | output -> {OUT}")


In [ ]:
import oil_futures_regime as ofr
from oil_futures_regime import (
    DEFAULT_MODEL_SPECS, DEFAULT_RULES, FAST_MODEL_SPECS, STABILITY_RULES,
    DEFAULT_STATE_FEATURES, build_commodity_state, conditional_reversion_table,
    format_stability, relative_gain_by_period, stability_model_specs, stability_report,
    aggregate_oos, baseline_in_mcs, best_model_summary, build_macro_features,
    common_uniforms, evaluate_rules, format_verdict,
    cts_state_summary, estimate_spread_half_life, exclude_period, filtered_states,
    fit_cts_model, fit_vecm_scenario_model, fit_state_dependent_spread, holm_adjust,
    influence_of_worst_origins, interaction_wald,
    johansen_rank_summary, load_eia_cushing, load_eia_wti_curve, load_long_oil_history,
    load_or_download, mcs_by_cell, mcs_summary,
    mean_vs_median_table, rolling_johansen_rank,
    paired_score_table, pit_summary, prepare_residual_pool, ranking_stability,
    roll_gap_diagnostics, rolling_oos_validate, rolling_state_dependent_oos,
    romano_wolf_stepdown, simulate,
    split_dev_holdout, var_es_report, var_johansen_summary,
)

SPECS = FAST_MODEL_SPECS if FAST else DEFAULT_MODEL_SPECS
print(f"oil_futures_regime {ofr.__version__}")
print(f"{len(SPECS)} models: {[s.name for s in SPECS]}")


## 2. Data

Read from the parquet cache written by `scripts/fetch_data.py`, so the run is
deterministic and independent of what Yahoo returns today. If the cache is
missing, the loader downloads once and writes it.

In [ ]:
if USE_SYNTHETIC:
    sys.path.insert(0, str(ROOT / "scripts"))  # set by the bootstrap cell
    from run_validation import synthetic_panel
    oil, state = synthetic_panel(break_date=BREAK_DATE if SYNTHETIC_BETA_BREAK else None)
    oos_start = oil.index[260]
else:
    oil = load_or_download("oil", OIL_TICKERS, START, END, ANCHOR,
                           cache_dir=CACHE, dropna="any")
    state = load_or_download("state", STATE_TICKERS, START, END, ANCHOR,
                             cache_dir=CACHE, dropna="all")
    oos_start = OOS_START

logp = np.log(oil[["WTI", "BRENT"]]).dropna()
macro = build_macro_features(state)
spread_obs = (logp["WTI"] - logp["BRENT"]).rename("log_spread")

# Stability diagnostics deliberately get more pre-2015 history without changing
# the 2013-start forecasting experiment. In synthetic mode the same panel is used.
if USE_SYNTHETIC:
    stability_logp = logp.copy()
else:
    try:
        stability_oil = load_long_oil_history(
            start=STABILITY_START, end=END, anchor=ANCHOR, cache_dir=CACHE, refresh=False
        )
        stability_logp = np.log(stability_oil[["WTI", "BRENT"]]).dropna()
    except Exception as exc:
        print(f"long stability history unavailable ({exc}); using the core sample")
        stability_logp = logp.copy()
spread_stability = (stability_logp["WTI"] - stability_logp["BRENT"]).rename("log_spread")

print(f"{len(logp)} weekly observations  {logp.index.min().date()} -> {logp.index.max().date()}")
print(f"stability sample: {len(stability_logp)} rows  "
      f"{stability_logp.index.min().date()} -> {stability_logp.index.max().date()}")
print(f"macro features: {list(macro.columns)}, {len(macro)} rows")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(oil.index, oil["WTI"], lw=1, label="WTI")
axes[0].plot(oil.index, oil["BRENT"], lw=1, label="Brent")
axes[0].set_title("Weekly front-month prices"); axes[0].legend()
axes[1].plot(spread_obs.index, spread_obs, lw=1, color="tab:red")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Observed WTI - Brent log spread")
plt.tight_layout(); plt.show()


### How contaminated is the spread?

`CL=F` and `BZ=F` are concatenated front-month series, not roll-adjusted.
Returns spanning a contract roll contain a jump that is not a tradeable return,
and it lands directly in the object the VECM and the state space model.

In [ ]:
flags = roll_gap_diagnostics(oil, z_threshold=4.0)
print(f"{len(flags)} weekly moves beyond 4 robust sigma")
display(flags.head(12))

if not flags.empty:
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(spread_obs.index, spread_obs, lw=0.9)
    for d in flags["date"].unique():
        ax.axvline(pd.Timestamp(d), color="tab:orange", alpha=0.4, lw=1)
    ax.set_title("Log spread with flagged extreme weeks (candidate roll artefacts)")
    plt.tight_layout(); plt.show()

## 3. What kind of system is this?

Before forecasting: one common stochastic trend or two? How fast does the
relative component revert? How much of the multi-week variance is common versus
relative? These three answers determine where forecastable structure can exist
at all.

In [ ]:
joh = var_johansen_summary(logp, var_lags=1)
vecm = fit_vecm_scenario_model(logp, coint_rank=1, k_ar_diff=1, deterministic="ci")
cts = fit_cts_model(logp, restrict_trend=True)
cts_free = fit_cts_model(logp, restrict_trend=False)
summ_restricted = cts_state_summary(cts)
summ_free = cts_state_summary(cts_free)
summ = summ_free  # v1.0 descriptive default: do not summarize a rejected restriction

beta_vecm = np.asarray(vecm.result.beta).ravel(); beta_vecm = beta_vecm / beta_vecm[0]
beta_cts_r = summ_restricted["beta_implied"] / summ_restricted["beta_implied"][0]
beta_cts_f = summ_free["beta_implied"] / summ_free["beta_implied"][0]
hl_direct = estimate_spread_half_life(spread_obs)

print("Johansen trace test")
display(pd.DataFrame({"trace": joh["johansen_trace"], "crit_95": joh["johansen_crit_95"]},
                     index=[f"r <= {i}" for i in range(len(joh["johansen_trace"]))]).round(3))
print(f"implied rank at 95%: {joh['rank95']}   VAR eigenvalue moduli: {np.round(joh['moduli'], 4)}")

print("\nCointegrating vector, normalized on WTI")
print(f"  VECM / Johansen : {np.round(beta_vecm, 4)}")
print(f"  CTS restricted  : {np.round(beta_cts_r, 4)}")
print(f"  CTS free        : {np.round(beta_cts_f, 4)}")

from scipy.stats import chi2
lr = 2 * (cts_free.params["loglike"] - cts.params["loglike"])
p_restrict = 1 - chi2.cdf(max(lr, 0), 1)
print(f"\nfree trend loading on Brent a = {cts_free.params['trend_loading_brent']:.4f}")
print(f"LR test of a = 1 (i.e. beta = [1,-1]): stat {lr:.3f}, p = {p_restrict:.4f}")

print("\nRelative-component mean reversion")
print(f"  raw WTI-Brent AR(1), for reference: rho = {hl_direct['rho']:.4f}, "
      f"half-life = {hl_direct['half_life_weeks']:.2f} weeks")
print(f"  CTS restricted latent state       : phi = {summ_restricted['spread_ar_coefficient']:.4f}, "
      f"half-life = {summ_restricted['spread_half_life_weeks']:.2f} weeks")
print(f"  CTS free latent state (v1.0)      : phi = {summ_free['spread_ar_coefficient']:.4f}, "
      f"half-life = {summ_free['spread_half_life_weeks']:.2f} weeks")
if p_restrict < 0.05:
    print("  -> restriction rejected: use the FREE latent state below; the restricted half-life is diagnostic only")


### The symmetric restriction is a hypothesis, not a convention

`CTS_*` imposes `beta = [1, -1]`, i.e. the common trend enters WTI and Brent
one-for-one. If the likelihood-ratio test above rejects that, the restriction
leaves a slice of a near-unit-root trend inside what the model calls the
"spread", which inflates its estimated persistence and biases the spread
forecasts. `CTSF_*` estimates the loading freely and is in the grid for exactly
this reason: where the restriction is rejected, read the latent half-life off the
free version and treat the restricted number as contaminated.

### Is the cointegration rank stable?

A rank read once on the full sample is a single draw. With near-unit-root data
the trace statistic for `r <= n-1` sits close to its critical value, so the
implied rank can flip between origins — and on WTI/Brent it does: the full-sample
test can reject `r <= 1` as well as `r = 0`, which would mean a *stationary*
system rather than one common trend.

That reading is implausible economically, but it should be tested rather than
argued away. Two things follow. First, the decision is re-run at every origin
below, so the rank imposed in the scenario models is justified by a distribution
rather than by one number. Second, `VARL_equal` and `VARL_time` — an unrestricted
VAR in log levels, which is exactly what a full-rank decision implies — are part
of the model grid, so the forecast comparison settles the question empirically.

In [ ]:
rank_table = rolling_johansen_rank(logp, oos_start=oos_start, stride=STRIDE)
display(johansen_rank_summary(rank_table))

if not rank_table.empty:
    n = logp.shape[1]
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(rank_table["origin"], rank_table[f"trace_r<={n-1}"], lw=1.2,
            label=f"trace statistic, r <= {n-1}")
    ax.axhline(rank_table[f"crit95_r<={n-1}"].iloc[0], color="tab:red", ls="--",
               lw=1, label="95% critical value")
    ax.set_title("The marginal rank decision through time "
                 "(above the line = reject one common trend)")
    ax.legend(); plt.tight_layout(); plt.show()

    share_rank1 = 100.0 * (rank_table["rank_95"] == 1).mean()
    print(f"rank = 1 at 95% in {share_rank1:.0f}% of origins; "
          f"rank = 1 at 99% in {100.0 * (rank_table['rank_99'] == 1).mean():.0f}%")

In [ ]:
# Where does multi-week OUTRIGHT variance come from under the free-loading CTSF?
# Measurement noise is omitted here; this decomposes latent state uncertainty.
phi = summ_free["spread_ar_coefficient"]
a_free = cts_free.params["trend_loading_brent"]
var_trend_1w = cts_free.params["sigma_eta"] ** 2
var_spread_unc = summ_free["spread_uncond_sd"] ** 2

rows = []
for h in [1, 4, 12, 20, 52]:
    v_tau = h * var_trend_1w
    v_s = var_spread_unc * (1 - phi ** (2 * h))
    # y_WTI = tau + .5 s ; y_Brent = a*tau - .5 s
    v_rel_each = 0.25 * v_s
    v_common_wti = v_tau
    v_common_brent = (a_free ** 2) * v_tau
    rows.append({
        "horizon_weeks": h,
        "sd_common_wti_%": 100 * np.sqrt(v_common_wti),
        "sd_common_brent_%": 100 * np.sqrt(v_common_brent),
        "sd_relative_each_%": 100 * np.sqrt(v_rel_each),
        "share_common_wti_%": 100 * v_common_wti / (v_common_wti + v_rel_each),
        "share_common_brent_%": 100 * v_common_brent / (v_common_brent + v_rel_each),
    })
decomp = pd.DataFrame(rows)
display(decomp.round(3))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(decomp["horizon_weeks"], decomp["share_common_wti_%"], marker="o", label="WTI")
ax.plot(decomp["horizon_weeks"], decomp["share_common_brent_%"], marker="o", label="Brent")
ax.set_xlabel("weeks ahead"); ax.set_ylabel("% latent variance from common trend")
ax.set_title("CTSF: common trend accumulates; stationary relative risk saturates")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
fs = filtered_states(cts_free, logp.index)
a_free = cts_free.params["trend_loading_brent"]
# beta=[a,-1] annihilates the common trend. Under the measurement equation,
# beta'y = .5*(a+1)*s + beta'eps, so put the filtered latent state on that scale.
cointegrating_obs = (a_free * logp["WTI"] - logp["BRENT"]).rename("beta_y")
relative_filtered = (0.5 * (a_free + 1.0) * fs["spread_filtered"]).rename("filtered_beta_y")
noise = cointegrating_obs.reindex(fs.index).values - relative_filtered.values

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(cointegrating_obs.index, cointegrating_obs, lw=0.8, alpha=0.5, label="observed beta'y")
axes[0].plot(fs.index, relative_filtered, lw=1.3, label="CTSF filtered relative component")
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_title("Free-loading cointegrating residual: observed vs Kalman-filtered"); axes[0].legend()
axes[1].plot(fs.index, noise, lw=0.7, color="tab:orange")
axes[1].set_title("Observed minus filtered: measurement noise / front-month artefacts")
plt.tight_layout(); plt.show()

print(f"sd(observed beta'y) {cointegrating_obs.std():.4f}   "
      f"sd(filtered) {relative_filtered.std():.4f}   sd(noise) {np.std(noise):.4f}")


## 3b. Is the cointegrating relation stable?

The full-sample cointegrating vector averages over a known structural change: the
repeal of the US crude-oil export ban, signed on **18 December 2015**. Before it,
US crude was largely landlocked and WTI traded at a variable, often wide discount
to Brent; afterwards, export arbitrage should have tied the two together. If
that is right, three separate things should have moved:

1. **the level of the relation** — the mean WTI discount should narrow;
2. **the slope** — the cointegrating coefficient itself;
3. **the spread dynamics** — tighter arbitrage should make deviations revert
   faster, and perhaps be less volatile.

Each is tested separately. The five primary known-date p-values are then Holm
corrected as one family, so a borderline result in one dimension is not promoted
merely because several related break hypotheses were tried.

**A calibration warning that shaped this section.** The textbook known-date test
(DOLS with an asymptotic chi-square) was simulated under *no break at all*, with a
spread persistence of 0.9 — close to what WTI/Brent shows. It rejected in 41% of
samples at a nominal 5%. On real data it would have "confirmed" a break at the
repeal almost regardless of the truth. Every p-value below is therefore calibrated
by an AR-sieve bootstrap under its own null (size 3-7% in simulation); the
asymptotic values are printed in brackets for reference only.

**Evidence hierarchy.** Read the known-date tests and the spread dynamics first.
The unknown-date search answers *whether* a break exists anywhere, but it locates
breaks poorly (median error about six months in simulation) and will be drawn
towards extreme episodes such as April 2020. The Kalman path is a picture: its
constant-slope test is conservative and has low power.


In [ ]:
%%time
beta_stab = stability_report(
    stability_logp, break_date=BREAK_DATE, n_boot=N_BOOT_STABILITY,
    rolling_window=ROLLING_WINDOW
)
print(format_stability(beta_stab))
print("\nPrimary known-date family: Holm FWER correction")
display(beta_stab["known_break_family"].round(4))


In [ ]:
bd = pd.Timestamp(BREAK_DATE)
kb, sw, tvp, sd = beta_stab["known_break"], beta_stab["sup_wald"], beta_stab["tvp"], beta_stab["spread_dynamics"]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# (a) the spread with pre/post means
ax = axes[0]
ax.plot(spread_stability.index, spread_stability, lw=0.8, color="tab:blue", alpha=0.8)
ax.hlines(sd["mean_pre"], spread_stability.index.min(), bd, color="tab:red", lw=2,
          label=f"mean pre  {sd['mean_pre']:+.3f}  (half-life {sd['half_life_pre']:.1f}w)")
ax.hlines(sd["mean_post"], bd, spread_stability.index.max(), color="tab:green", lw=2,
          label=f"mean post {sd['mean_post']:+.3f}  (half-life {sd['half_life_post']:.1f}w)")
ax.axvline(bd, color="k", ls="--", lw=1)
ax.set_title("WTI - Brent log spread, before and after the export-ban repeal")
ax.legend(loc="lower right")

# (b) the slope through time: Kalman (smoothed) and rolling DOLS
ax = axes[1]
p = tvp["paths"]
ax.plot(p.index, p["slope_smoothed"], color="tab:purple", lw=1.4, label="Kalman slope (smoothed)")
ax.fill_between(p.index, p["slope_smoothed"] - 1.96 * p["slope_smoothed_se"],
                p["slope_smoothed"] + 1.96 * p["slope_smoothed_se"], color="tab:purple", alpha=0.15)
roll = beta_stab["rolling"]
if not roll.empty:
    ax.plot(roll.index, roll["beta"], color="tab:orange", lw=1.2,
            label=f"rolling DOLS, {ROLLING_WINDOW}w window")
    ax.fill_between(roll.index, roll["lo"], roll["hi"], color="tab:orange", alpha=0.12)
ax.axhline(1.0, color="k", lw=0.6)
ax.axvline(bd, color="k", ls="--", lw=1)
ax.set_title("Cointegrating slope through time (descriptive)")
ax.legend(loc="lower right")

# (c) where the data would put a break
ax = axes[2]
ax.plot(sw["path"].index, sw["path"].values, lw=1.2, color="tab:gray", label="Wald statistic by date")
ax.axhline(sw["boot_95"], color="tab:red", ls=":", lw=1.2, label="bootstrap 95% of the sup under no break")
ax.axvline(bd, color="k", ls="--", lw=1, label=f"repeal {bd.date()}")
ax.axvline(sw["break_date_hat"], color="tab:blue", lw=1.2, label=f"LS estimate {sw['break_date_hat'].date()}")
lo, hi = sw["location_range90_under_h0"]
if pd.notna(lo):
    ax.axvspan(lo, hi, color="tab:blue", alpha=0.08,
               label="where the estimator lands 90% of the time if the repeal is the break")
ax.set_title("Unknown-date search: existence and (imprecise) location")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()


**How to read this.**

- A significant **level** jump with a narrowing discount is the export-arbitrage
  story. A significant **speed** change with a shorter post-repeal half-life is
  its dynamic counterpart.
- A **slope** rejection *next to* a level jump deserves caution. In simulation,
  a gradual transition to a new mean — which is what a change in persistence
  produces — leaks partly into the slope coefficient of a step-dummy regression.
  A slope rejection on its own is a different and stronger claim.
- The **location** estimate is a range, not a date. The shaded band shows how far
  from the repeal the estimator would typically land even if the repeal were the
  true break; an estimate inside or near it is compatible with the hypothesis.
- The **Kalman** path is descriptive in both directions: it has little power
  against genuine slope drift, and it can reject under volatility clustering
  alone (p = 0.026 on a synthetic GARCH panel with no break). Treat a Kalman
  rejection as a prompt to look, not as evidence, unless the bootstrap tests agree.
- The original core sample had only about three years of pre-repeal data. v1.0
  therefore runs this section on a dedicated longer history (default 2008 start)
  while leaving the 2013 forecasting experiment untouched. Even on the longer
  sample, failure to reject is evidence of limited power, not proof of no break.
- In the short sample, the
  pre-repeal mean itself is uncertain by roughly ±0.03 (the standard deviation
  of the sample mean across 30 simulated panels). A real economic shift in the
  level can fail to reach significance for that reason alone.
- None of this says whether the break *matters for forecasting*. That is §8b.


## 4. The rolling ablation

Every model is re-estimated at every origin from data available at or before it:
AR order selection, VECM, GARCH filter, state-space MLE, macro standardization
and kernel bandwidth. An origin enters the panel only if **all** models produced
a forecast, and all models share one set of random draws at each origin.

This is the expensive cell — roughly 15 minutes for the full grid.

In [ ]:
%%time
vo = rolling_oos_validate(
    log_prices=logp,
    macro_features=macro,
    horizons=HORIZONS,
    oos_start=oos_start,
    stride=STRIDE,
    n_sim=N_SIM,
    half_life=HALF_LIFE,
    min_macro_ess_fraction=MIN_MACRO_ESS_FRACTION,
    model_specs=SPECS,
    seed=SEED,
)
results = vo.results

print(f"\norigins kept {len(vo.origins_used)}   dropped {len(vo.origins_dropped)}")
print(f"balanced panel: {vo.is_balanced}")
print(f"effective independent observations at h={max(HORIZONS)}: "
      f"~{len(vo.origins_used) * STRIDE / max(HORIZONS):.0f}")
if not vo.failures.empty:
    print("\nFailures (logged, never silently skipped):")
    display(vo.failures.head(10))

## 5. Leaderboard and pairwise significance

In [ ]:
summary = aggregate_oos(results)
for variable in ["WTI", "BRENT", "SPREAD"]:
    print(f"\n{variable} — mean CRPS")
    display(summary[summary["variable"] == variable]
            .pivot(index="model", columns="horizon_weeks", values="mean_crps").round(5))

In [ ]:
sig = paired_score_table(results, baseline=BASELINE, stride=STRIDE, n_boot=N_BOOT)
best = best_model_summary(sig)
display(best.round(4))

winners = sig[(sig["significant_5pct"]) & (sig["improvement_pct"] > 0)]
print(f"\nPairwise-significant improvements over {BASELINE} (uncorrected):")
display(winners[["variable", "horizon_weeks", "model", "improvement_pct",
                 "dm_stat", "dm_pvalue", "boot_lo", "boot_hi"]].round(4)
        if not winners.empty else "none")

## 6. Multiplicity

The table above is uncorrected. Across the full grid there are
`n_models × n_variables × n_horizons` comparisons; at 5% a proportional number of
spurious winners is expected by chance. Two corrections follow.

**Model Confidence Set** (Hansen–Lunde–Nason) returns the *set* of models that
cannot be separated from the best. With fewer than 20 effectively independent
observations at long horizons, an honest answer is usually a set rather than a
point.

**Romano–Wolf stepdown** controls the probability of *any* false rejection
across the family tested against the baseline, and is more powerful than
Bonferroni because the bootstrap keeps the dependence between statistics.

In [ ]:
%%time
mcs_table = mcs_by_cell(results, score="crps", alpha=MCS_ALPHA,
                        stride=STRIDE, n_boot=N_BOOT, seed=SEED)
mcs_view = mcs_summary(mcs_table)
display(mcs_view)

In [ ]:
rw_frames = []
for (variable, horizon), g in results.groupby(["variable", "horizon_weeks"], observed=True):
    wide = g.pivot(index="origin", columns="model", values="crps").dropna(how="any")
    rw = romano_wolf_stepdown(wide, baseline=BASELINE, horizon=int(horizon),
                              stride=STRIDE, n_boot=N_BOOT, seed=SEED)
    rw.insert(0, "horizon_weeks", int(horizon)); rw.insert(0, "variable", variable)
    rw_frames.append(rw)
romano = pd.concat(rw_frames, ignore_index=True)

surv = romano[romano["reject_at_5pct"]]
print("Survives within-cell familywise correction against the baseline:")
display(surv[["variable", "horizon_weeks", "model", "improvement_pct",
              "p_raw", "p_familywise"]].round(4) if not surv.empty
        else "Nothing survives familywise correction.")

n_raw = int((romano["p_raw"] < 0.05).sum())
print(f"\nuncorrected rejections: {n_raw}   familywise-controlled: {len(surv)}"
      f"   out of {len(romano)} reported model-cell comparisons (FWER controlled within each cell)")

## 7. Robustness

Two checks that a mean-CRPS table hides: whether a few extreme origins carry the
result, and whether the ranking survives removing them.

In [ ]:
mvm = mean_vs_median_table(results)
suspects = mvm[mvm["rank_gap"] < -2]
print("Models ranking much better on the mean than on the median")
print("(i.e. winning on a few extreme origins rather than on typical ones):")
display(suspects.head(15).round(5) if not suspects.empty else "none")

In [ ]:
infl = influence_of_worst_origins(results, top_k=3)
worst_cell = infl[(infl["variable"] == "WTI") & (infl["horizon_weeks"] == max(HORIZONS))]
display(worst_cell[["model", "mean_full", f"mean_excl_worst_3",
                    "share_from_worst_pct", "rank_full", "rank_trimmed",
                    "worst_origins"]].round(4))
print("share_from_worst_pct: how much of the average loss comes from 3 origins out of "
      f"{len(vo.origins_used)}")

In [ ]:
no_crisis = exclude_period(results, *CRISIS_WINDOW, on="target_date")
print(f"excluding {CRISIS_WINDOW[0]} -> {CRISIS_WINDOW[1]}: "
      f"{results['origin'].nunique()} -> {no_crisis['origin'].nunique()} origins")

stab = ranking_stability(results, no_crisis, label="no_crisis")
moved = stab[stab["rank_change"].abs() >= 2]
print("\nModels whose rank moves by 2 or more once the crisis window is dropped:")
display(moved.round(2) if not moved.empty else "ranking is stable")

if len(no_crisis) and no_crisis["origin"].nunique() > 20:
    sig_nc = paired_score_table(no_crisis, baseline=BASELINE, stride=STRIDE, n_boot=N_BOOT)
    w_nc = sig_nc[(sig_nc["significant_5pct"]) & (sig_nc["improvement_pct"] > 0)]
    print("\nSignificant winners excluding the crisis window:")
    display(w_nc[["variable", "horizon_weeks", "model", "improvement_pct", "dm_pvalue"]].round(4)
            if not w_nc.empty else "none")

## 8. Late development/validation split

The rolling experiment is pseudo-out-of-sample, but this 2023-2024 block has now
been inspected while later versions of the framework were developed. It is
therefore **not an untouched holdout**. We use it only as a ranking-stability
check between earlier and later forecast origins.

The genuinely external test is separate: `scripts/run_forward_holdout.py` starts
with the first origin after 7 March 2025 and applies the frozen v1.0 procedure.
Do not change model/rule selection after inspecting that output.


In [ ]:
dev, hold = split_dev_holdout(results, VALIDATION_START, on="origin")
print(f"development: {dev['origin'].nunique()} origins "
      f"({pd.to_datetime(dev['origin']).min().date()} -> {pd.to_datetime(dev['origin']).max().date()})")
print(f"late validation: {hold['origin'].nunique()} origins "
      f"({pd.to_datetime(hold['origin']).min().date()} -> {pd.to_datetime(hold['origin']).max().date()})")

if hold["origin"].nunique() >= 12:
    dev_rank = aggregate_oos(dev).rename(columns={"mean_crps": "crps_dev"})
    hold_rank = aggregate_oos(hold).rename(columns={"mean_crps": "crps_late"})
    comp = dev_rank.merge(hold_rank, on=["model", "variable", "horizon_weeks"])
    comp["rank_dev"] = comp.groupby(["variable", "horizon_weeks"])["crps_dev"].rank()
    comp["rank_late"] = comp.groupby(["variable", "horizon_weeks"])["crps_late"].rank()
    for variable in comp["variable"].unique():
        sub = comp[comp["variable"] == variable]
        rho = sub.groupby("horizon_weeks").apply(
            lambda g: g["rank_dev"].corr(g["rank_late"], method="spearman"))
        print(f"\n{variable}: Spearman rank correlation early vs late validation by horizon")
        print(rho.round(3).to_string())
    display(comp[comp["variable"] == "SPREAD"][
        ["model", "horizon_weeks", "crps_dev", "crps_late", "rank_dev", "rank_late"]
    ].sort_values(["horizon_weeks", "crps_late"]).round(5).head(20))
else:
    print("\nLate validation block too small to evaluate; widen the sample or move VALIDATION_START earlier.")


## 8b. Does the instability matter for forecasting?

A break can be statistically detectable and still irrelevant for forecasting —
the cointegration rank in §5 is the precedent. The operational question is
whether estimating only on post-repeal data, or on a moving window, improves
spread forecasts over the expanding window used everywhere else.

This runs as a **separate, pre-specified hypothesis family** because it asks a
different question — whether parameter-instability remedies improve forecasts —
not because adding the variants would weaken the primary confidence sets.

**What to expect, from simulation.** With a genuine repeal-shaped break planted
in synthetic data, post-break estimation improved spread CRPS by 25-32% at
origins within two years of the break, then faded to noise as the expanding
window filled with post-break observations. Pooled over 2018-2024 origins the
effect was slightly *negative*. Since the out-of-sample origins here start in
2018, the pooled test mostly measures the phase in which the break no longer
matters, so **a DROP does not mean "no break"** — it means an expanding window
had already absorbed it. The gain by origin year, below, is where a transient
effect shows up.


In [ ]:
%%time
stab_specs = stability_model_specs(break_date=BREAK_DATE, window=ROLLING_WINDOW)
print([s.name for s in stab_specs])

vo_stab = rolling_oos_validate(
    log_prices=logp, macro_features=macro, horizons=HORIZONS, oos_start=oos_start,
    stride=STRIDE, n_sim=N_SIM, half_life=HALF_LIFE, model_specs=stab_specs,
    variables=("SPREAD",), seed=SEED, verbose=False,
)
print(f"origins kept {len(vo_stab.origins_used)}   dropped {len(vo_stab.origins_dropped)}"
      f"   balanced {vo_stab.is_balanced}")
if not vo_stab.failures.empty:
    display(vo_stab.failures.head())

stab_summary = aggregate_oos(vo_stab.results)
print("\nSPREAD mean CRPS")
display(stab_summary.pivot(index="model", columns="horizon_weeks", values="mean_crps").round(5))

In [ ]:
stab_rules, stab_detail = evaluate_rules(vo_stab.results, rules=STABILITY_RULES,
                                         stride=STRIDE, n_boot=N_BOOT, seed=SEED)
for r in stab_rules.itertuples():
    print(f"[{r.decision}] {r.rule}: {r.challenger} vs {r.incumbents}   "
          f"significant in {r.n_significant}/{r.n_cells} after correction   "
          f"median {r.median_improvement_pct:+.2f}%")

print("\nGain from post-repeal estimation by origin year (positive = post-repeal better)")
gain_vecm = relative_gain_by_period(vo_stab.results, "VECM_post", "VECM_equal")
display(gain_vecm.round(1))
gain_ctsf = relative_gain_by_period(vo_stab.results, "CTSF_post", "CTSF_equal")
print("Same for the free-beta state space")
display(gain_ctsf.round(1))

In [ ]:
# A short, paste-ready summary of the stability evidence for the write-up.
kb, sw, sd, tvp = beta_stab["known_break"], beta_stab["sup_wald"], beta_stab["spread_dynamics"], beta_stab["tvp"]
lines = [
    "# Stability of the WTI/Brent cointegrating relation",
    "",
    f"Known date: {pd.Timestamp(BREAK_DATE).date()} (US crude-oil export ban repealed). "
    "All p-values are AR-sieve bootstrap calibrated.",
    "",
    "| quantity | pre | post | p-value |",
    "|---|---:|---:|---:|",
    f"| level of the relation | {kb['level_pre']:+.4f} | {kb['level_post']:+.4f} | {kb['p_level_boot']:.4f} |",
    f"| cointegrating slope | {kb['slope_pre']:.4f} | {kb['slope_post']:.4f} | {kb['p_slope_boot']:.4f} |",
    f"| spread mean | {sd['mean_pre']:+.4f} | {sd['mean_post']:+.4f} | {sd['p_joint_boot']:.4f} |",
    f"| spread persistence (rho) | {sd['rho_pre']:.4f} | {sd['rho_post']:.4f} | {sd['p_speed_boot']:.4f} |",
    f"| spread half-life (weeks) | {sd['half_life_pre']:.2f} | {sd['half_life_post']:.2f} | |",
    f"| weekly spread volatility | {sd['vol_pre']:.4f} | {sd['vol_post']:.4f} | {sd['p_vol_boot']:.4f} |",
    "",
    f"Unknown-date search: sup-Wald {sw['sup_wald']:.2f}, bootstrap p = {sw['p_value_bootstrap']:.4f}; "
    f"least-squares date {sw['break_date_hat'].date()} "
    f"({sw['estimate_minus_hypothesis_weeks']:+.0f} weeks from the repeal); "
    f"p-value that the break is at the repeal {sw['location_p_value']:.4f}.",
    "",
    f"Kalman constant-slope LR {tvp['lr_constant_slope']:.2f}, "
    f"boundary-corrected p = {tvp['p_constant_slope']:.4f} (descriptive; low power).",
    "",
    "## Primary known-date family: Holm FWER correction",
    "",
    beta_stab["known_break_family"].round(4).to_markdown(index=False),
    "",
    "## Forecasting consequence (SPREAD, familywise-corrected within each rule)",
    "",
    stab_rules[["rule", "decision", "n_significant", "n_cells",
                "median_improvement_pct"]].round(3).to_markdown(index=False),
    "",
    "Gain from post-repeal estimation by origin year, VECM (%):",
    "",
    gain_vecm.round(1).to_markdown(),
]
(OUT / "beta_stability.md").write_text("\n".join(lines))
vo_stab.results.to_csv(OUT / "stability_forecast_rows.csv.gz", index=False, compression="gzip")
stab_rules.to_csv(OUT / "stability_rules.csv", index=False)
print("written", OUT / "beta_stability.md")


## 9. Calibration and tail backtests

In [ ]:
cov = summary.pivot_table(index="model", columns=["variable", "horizon_weeks"],
                          values="coverage_90").round(3)
print("90% interval coverage (target 0.90)")
display(cov)

pit = pit_summary(results)
print("\nWorst PIT uniformity (descriptive: overlapping PITs are dependent)")
display(pit.sort_values("ks_uniform_stat", ascending=False).head(10).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
show = [BASELINE] + [m for m in ["AR_macro", "CTS_time"] if m in set(results["model"])]
for ax, model in zip(axes, show[:3]):
    sel = results[(results["model"] == model) & (results["variable"] == "WTI") &
                  (results["horizon_weeks"] == HORIZONS[min(1, len(HORIZONS)-1)])]
    ax.hist(sel["pit"], bins=10, range=(0, 1), edgecolor="white")
    ax.axhline(len(sel) / 10, color="k", ls="--", lw=1)
    ax.set_title(f"PIT — {model}, WTI")
plt.tight_layout(); plt.show()

In [ ]:
risk = var_es_report(results, stride=STRIDE, alphas=(0.05, 0.01))
r5 = risk[(risk["alpha"] == 0.05) & (~risk["underpowered"])]
print("5% VaR/ES backtests on the overlap-thinned panel")
display(r5[["variable", "horizon_weeks", "model", "n_used", "exception_rate",
            "kupiec_p", "christoffersen_ind_p", "christoffersen_cc_p",
            "es_z2", "es_z2_p"]].round(4).head(25))

rejected = r5[r5["christoffersen_cc_p"] < 0.05]
print(f"\nRejected by conditional coverage at 5%: {len(rejected)} of {len(r5)} tested cells")
display(rejected[["variable", "horizon_weeks", "model", "exception_rate",
                  "christoffersen_cc_p"]].round(4) if not rejected.empty else "none")
print(f"\nUnderpowered cells excluded: {int(risk['underpowered'].sum())} "
      "(fewer than 2 expected exceptions — a pass there means nothing)")

## 10. Verdict

For v1.0 the rules are **fixed in code and applied mechanically**. Earlier
versions of the project refined the rule after inspecting development results,
so this is deliberately called a *pre-specified v1.0 rule*, not a preregistration.

**Multiplicity inside each rule.** Each rule is tested across twelve cells
(3 variables x 4 horizons). Declaring `KEEP` on a single cell at p < 0.05 is too
lenient: about 0.6 spurious wins per rule are expected by chance. All cells of a
rule share the same forecast origins, so their loss differentials form a panel
and a Romano-Wolf stepdown corrects across them while preserving dependence. A
rule is kept only if some cell has positive improvement with familywise-adjusted
p < 5%.

**No post-hoc winner.** Instead of choosing a winner and testing it on the same
data, ask whether the random walk is **excluded from the 90% Model Confidence
Set**. If it is, something beats it without requiring a post-hoc model choice.

Each rule also compares its challenger against the *best* simpler alternatives.
A `DROP` is a result, not a problem.


In [ ]:
rule_summary, rule_detail = evaluate_rules(
    results, rules=DEFAULT_RULES, stride=STRIDE, n_boot=N_BOOT, seed=SEED
)
baseline_table = baseline_in_mcs(mcs_table, baseline=BASELINE)

print(format_verdict(rule_summary, baseline_table, rules=DEFAULT_RULES, baseline=BASELINE))

In [ ]:
# Per-cell detail behind the verdict: raw and familywise-adjusted p-values on the
# same one-sided statistic, plus the incumbent that was binding in each cell.
cols = ["rule", "variable", "horizon_weeks", "incumbent", "improvement_pct",
        "p_raw", "p_familywise", "passes"]
display(rule_detail[cols].round(4))

kept = rule_summary[rule_summary["decision"] == "KEEP"]["rule"].tolist()
dropped = rule_summary[rule_summary["decision"] == "DROP"]["rule"].tolist()
print(f"KEEP: {kept if kept else 'nothing'}")
print(f"DROP: {dropped if dropped else 'nothing'}")

n_excl = int(baseline_table["something_beats_baseline"].sum())
print(f"\nThe baseline is excluded from the 90% MCS in {n_excl} of "
      f"{len(baseline_table)} cells.")
if n_excl:
    where = ", ".join(f"{r.variable}/h{r.horizon_weeks}" for r in
                      baseline_table[baseline_table["something_beats_baseline"]].itertuples())
    print(f"Those cells are: {where}")

In [ ]:
# Persist everything
results.to_csv(OUT / "forecast_rows.csv.gz", index=False, compression="gzip")
summary.to_csv(OUT / "summary.csv", index=False)
sig.to_csv(OUT / "significance.csv", index=False)
best.to_csv(OUT / "best_models.csv", index=False)
mcs_table.to_csv(OUT / "mcs.csv", index=False)
mcs_view.to_csv(OUT / "mcs_summary.csv", index=False)
romano.to_csv(OUT / "romano_wolf.csv", index=False)
rule_summary.to_csv(OUT / "decision_rule_summary.csv", index=False)
rule_detail.to_csv(OUT / "decision_rule_cells.csv", index=False)
baseline_table.to_csv(OUT / "baseline_vs_mcs.csv", index=False)
rank_table.to_csv(OUT / "johansen_rank_stability.csv", index=False)
mvm.to_csv(OUT / "mean_vs_median.csv", index=False)
infl.to_csv(OUT / "worst_origin_influence.csv", index=False)
pit.to_csv(OUT / "pit.csv", index=False)
risk.to_csv(OUT / "var_es.csv", index=False)
vo.failures.to_csv(OUT / "failures.csv", index=False)
(OUT / "config.json").write_text(json.dumps(vo.config, indent=2, default=str))
print("written to", OUT)
print(sorted(p.name for p in OUT.iterdir()))

## 11. Experimental commodity-specific state: WTI curve + Cushing inventories

This branch asks a separate oil-market question: whether WTI-Brent spread
mean-reversion changes with public EIA WTI C1-C4 curve state and
publication-lagged Cushing inventories. It is **not part of the validated v1.0
headline** and is disabled by default.

The current specification is deliberately interpretable:

`Delta s[t+1] = c + lambda*s[t] + theta'z[t] + delta'(s[t]*z[t]) + eps[t+1]`.

The real-data fit shows strong in-sample interaction evidence, but recursively
iterating the unconstrained state-dependent persistence can become unstable at
extreme states over multi-week horizons. The next specification should therefore
use direct-horizon/local-projection forecasts or a bounded persistence map.

The EIA C1-C4 history ends in April 2024, so even after that fix this remains a
historical mechanism test rather than a live-signal claim.


In [ ]:
commodity_results = None
if RUN_COMMODITY_STATE and not USE_SYNTHETIC:
    try:
        eia_curve = load_eia_wti_curve(CACHE, refresh=False)
        cushing = load_eia_cushing(CACHE, refresh=False)
        oil_state = build_commodity_state(eia_curve, cushing, inventory_release_lag_weeks=1)
        spread_for_state = (logp["WTI"] - logp["BRENT"]).rename("SPREAD")
        common_end = min(oil_state.dropna().index.max(), spread_for_state.index.max())

        sd_model = fit_state_dependent_spread(
            spread_for_state.loc[:common_end], oil_state.loc[:common_end],
            features=DEFAULT_STATE_FEATURES,
        )
        print(f"commodity-state common sample ends {common_end.date()} | n={sd_model.nobs}")
        print("joint HAC Wald on spread x state interactions:", interaction_wald(sd_model))
        display(conditional_reversion_table(sd_model).round(4))

        commodity_results = rolling_state_dependent_oos(
            spread_for_state, oil_state, features=DEFAULT_STATE_FEATURES,
            horizons=HORIZONS, oos_start=oos_start, stride=STRIDE,
            n_sim=N_SIM, seed=SEED,
        )
        commodity_sig = paired_score_table(
            commodity_results, baseline="SPREAD_AR1", stride=STRIDE,
            n_boot=N_BOOT, seed=SEED,
        )
        commodity_ch = commodity_sig[commodity_sig["model"] == "SD_EC"].copy()
        commodity_holm = holm_adjust({
            f"h{int(r.horizon_weeks)}": r.dm_pvalue for r in commodity_ch.itertuples()
        })
        print(f"balanced commodity-state OOS origins: {commodity_results['origin'].nunique()}")
        display(commodity_ch[["horizon_weeks", "improvement_pct", "dm_pvalue"]].round(4))
        print("Holm correction across horizons")
        display(commodity_holm.round(4))

        cs_out = ROOT / "reports" / "commodity_state"
        cs_out.mkdir(parents=True, exist_ok=True)
        commodity_results.to_csv(cs_out / "forecast_rows.csv.gz", index=False, compression="gzip")
        commodity_sig.to_csv(cs_out / "significance.csv", index=False)
        commodity_holm.to_csv(cs_out / "horizon_family_holm.csv", index=False)
        conditional_reversion_table(sd_model).to_csv(cs_out / "conditional_reversion.csv", index=False)
    except Exception as exc:
        print("commodity-state branch not run:", exc)
        print("Fetch/cache the public inputs first with: python scripts/fetch_data.py")
else:
    print("commodity-state branch skipped in synthetic core mode; use scripts/run_commodity_state.py --synthetic for the offline DGP check")


## 12. Genuine forward external validation

The 2023-2024 split above is development/validation data. The external block is
**post-7-March-2025** and is intentionally kept outside this automatic notebook
run so that seeing it does not trigger another round of model selection.

After refreshing the latest caches, run once:

```bash
python scripts/fetch_data.py
python scripts/run_forward_holdout.py --out reports/forward_holdout
```

The script re-estimates the same frozen v1.0 model grid at each new origin using
only information available then. Once inspected, treat that block as spent:
report it, do not tune to it.


## What to write up

`reports/RESULTS_wti_brent.md` is the written study for the executed core run.
The three claims that carry it are:

1. **Model Confidence Sets, not winner labels.** Where most models survive, the
   data cannot separate them.
2. **Familywise inference.** Spread improvements at 12/20 weeks survive the
   multiple-model correction; most other apparent edges do not.
3. **Robustness without relabelling validation data.** The spread ranking survives
   crisis exclusion and the late development/validation split. The genuinely
   external evidence is the frozen post-7-March-2025 forward block.

The structural-stability section is a separate hypothesis family, with Holm FWER
control over the five known-date tests. The commodity-state branch then asks a
new economic question — *when* relative-price convergence changes with the oil
curve / inventories — rather than adding another generic forecasting model.

Next material data upgrade: full contract-level WTI + Brent curves; after that,
scenario-to-P&L for Strat or a cost-aware relative-value decision layer for QR.
